# Caravan: preliminary EDA

Caravan (Kratzert et al., 2023) is a global, harmonized set of catchment attributes, ERA5-Land meteorological forcing, and daily streamflow for gauged basins, grouped into sub-datasets (`camels`, `camelsaus`, `hysets`, `lamah`, ...).

**Setup notes**
- The GitHub repo (`external/Caravan`, a git submodule) only holds the *processing* code (`caravan_utils.py`, Earth Engine notebooks). The *data* lives on Zenodo. There is no pip package; `pip install caravan` is an unrelated project.
- Download the **CSV release** (<https://zenodo.org/records/15530021>), unzip it, and put it at `data/Caravan/` (git-ignored), or set the `CARAVAN_DIR` environment variable.
- Expected layout: `attributes/<subset>/attributes_{caravan,hydroatlas,other}_<subset>.csv` and `timeseries/csv/<subset>/<gauge_id>.csv`.

In [ ]:
import os, sys, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
CARAVAN_DIR = Path(os.environ.get("CARAVAN_DIR", REPO / "data" / "Caravan"))
ATTR_DIR = CARAVAN_DIR / "attributes"
TS_DIR = CARAVAN_DIR / "timeseries" / "csv"
assert ATTR_DIR.exists(), f"Caravan data not found at {CARAVAN_DIR}. See setup notes above."

# Make the submodule's helper code importable (processing utilities, not a data loader)
sys.path.insert(0, str(REPO / "external" / "Caravan" / "code"))
try:
    import caravan_utils  # needs numba, timezonefinder, xarray, tqdm
    print("caravan_utils imported")
except ImportError as e:
    print("caravan_utils not importable (optional for EDA):", e)

SUBSETS = sorted(p.name for p in ATTR_DIR.iterdir() if p.is_dir())
print(CARAVAN_DIR, "\nsub-datasets:", SUBSETS)

## 1. Static attributes

In [ ]:
def load_attributes(subset):
    frames = []
    for kind in ("caravan", "hydroatlas", "other"):
        f = ATTR_DIR / subset / f"attributes_{kind}_{subset}.csv"
        if f.exists():
            frames.append(pd.read_csv(f, index_col="gauge_id"))
    return pd.concat(frames, axis=1).assign(subset=subset)

attrs = pd.concat([load_attributes(s) for s in SUBSETS])
attrs = attrs.loc[:, ~attrs.columns.duplicated()]
print(attrs.shape)
attrs.head()

In [ ]:
attrs["subset"].value_counts().rename("n_basins").to_frame()

In [ ]:
# Missingness per attribute (top 15) and dtype mix
miss = attrs.isna().mean().sort_values(ascending=False)
print(attrs.dtypes.value_counts().to_string(), "\n")
miss[miss > 0].head(15).to_frame("frac_missing")

In [ ]:
# Gauge locations
lat, lon = "gauge_lat", "gauge_lon"
if {lat, lon} <= set(attrs.columns):
    fig, ax = plt.subplots(figsize=(12, 5.5))
    sns.scatterplot(data=attrs, x=lon, y=lat, hue="subset", s=6, linewidth=0, ax=ax)
    ax.set(title="Caravan gauges", xlabel="lon", ylabel="lat")
    ax.legend(markerscale=3, fontsize=8, ncol=2)
    plt.show()

In [ ]:
# Key catchment descriptors (log scale where heavy-tailed)
cols = [c for c in ["area", "p_mean", "pet_mean", "aridity_ERA5_LAND", "frac_snow", "ele_mt_sav", "slp_dg_sav"] if c in attrs.columns]
fig, axes = plt.subplots(2, (len(cols) + 1) // 2, figsize=(15, 7))
for ax, c in zip(axes.ravel(), cols):
    x = attrs[c].dropna()
    log = c in ("area", "aridity_ERA5_LAND") and (x > 0).all()
    sns.histplot(np.log10(x) if log else x, bins=50, ax=ax)
    ax.set_xlabel(f"log10({c})" if log else c)
for ax in axes.ravel()[len(cols):]:
    ax.axis("off")
plt.tight_layout(); plt.show()
attrs[cols].describe().T

In [ ]:
# Climate / physiography correlations among the numeric Caravan-derived attributes
num = attrs[cols].select_dtypes("number")
sns.heatmap(num.corr(method="spearman"), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Spearman correlation"); plt.show()

## 2. Time series

In [ ]:
def ts_path(gauge_id):
    return TS_DIR / gauge_id.split("_")[0] / f"{gauge_id}.csv"

def load_ts(gauge_id):
    return pd.read_csv(ts_path(gauge_id), parse_dates=["date"], index_col="date")

# Look at one basin first
example = attrs.index[0]
ts = load_ts(example)
print(example, ts.index.min().date(), "->", ts.index.max().date(), ts.shape)
display(ts.head())
print("columns:", list(ts.columns))

In [ ]:
# Per-basin record summary on a random sample (reading every file can be slow)
random.seed(780)
sample = random.sample(list(attrs.index), k=min(200, len(attrs)))
rows = []
for g in sample:
    if not ts_path(g).exists():
        continue
    d = load_ts(g)
    q = d["streamflow"]
    rows.append(dict(gauge_id=g, start=d.index.min(), end=d.index.max(), n_days=len(d),
                     q_missing=q.isna().mean(), q_neg=(q < 0).sum(), q_zero=(q == 0).mean(),
                     q_mean=q.mean(), q_max=q.max()))
rec = pd.DataFrame(rows).set_index("gauge_id")
rec.describe().T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(rec["n_days"] / 365.25, bins=30, ax=axes[0]); axes[0].set_xlabel("record length (years)")
sns.histplot(rec["q_missing"], bins=30, ax=axes[1]); axes[1].set_xlabel("fraction missing streamflow")
sns.histplot(rec["start"].dt.year, bins=30, ax=axes[2]); axes[2].set_xlabel("record start year")
plt.tight_layout(); plt.show()

In [ ]:
# Streamflow + precipitation for the example basin, plus seasonal cycle and flow-duration curve
precip = next((c for c in ts.columns if c.startswith("total_precipitation_sum")), None)
temp = next((c for c in ts.columns if c.startswith("temperature_2m_mean")), None)

fig, axes = plt.subplots(2, 2, figsize=(15, 8))
ts["streamflow"].plot(ax=axes[0, 0], lw=.6); axes[0, 0].set(title=f"{example}: streamflow", ylabel="mm/d")
if precip: ts[precip].plot(ax=axes[0, 1], lw=.4, color="tab:blue"); axes[0, 1].set(title=precip)
clim = ts.groupby(ts.index.dayofyear)[["streamflow"] + [c for c in (precip, temp) if c]].mean()
clim["streamflow"].plot(ax=axes[1, 0]); axes[1, 0].set(title="mean daily streamflow by day of year")
q = np.sort(ts["streamflow"].dropna().values)[::-1]
axes[1, 1].semilogy(100 * np.arange(1, len(q) + 1) / len(q), np.clip(q, 1e-4, None))
axes[1, 1].set(title="flow-duration curve", xlabel="exceedance %")
plt.tight_layout(); plt.show()

In [ ]:
# Forcing/target variables: availability and a quick lagged-correlation check with streamflow
print(ts.isna().mean().sort_values(ascending=False).head(10).to_string(), "\n")
if precip:
    lags = range(0, 8)
    print({k: round(ts["streamflow"].corr(ts[precip].shift(k)), 3) for k in lags})

## 3. Notes / open questions for review
- Which sub-datasets and how many basins do we want for the unmonitored-basin (PUB) experiment?
- How should we treat basins with short records or high streamflow missingness (see section 2)?
- Confirm column names against the downloaded release; some cells select columns defensively and silently skip if absent.